# Mestres do XGH — Aula 7
## Dicionários e JSON: dado com nome

**Shift + Enter** roda a célula.

Hoje vocês passam a ler a língua em que **toda API de IA fala**.

---
# 1. A pergunta que ficou de sábado passado

Você carregava duas listas em paralelo, amarradas pelo índice. Eu falei que isso quebra
no dia em que alguém ordenar uma e esquecer a outra.

Vamos ver acontecer:

In [ ]:
nomes = ["Ana", "Bruno", "Carla"]
notas = [8.5, 4.0, 9.5]

notas.sort(reverse=True)      # ordenei só as notas

for i in range(len(nomes)):
    print(f"{nomes[i]:6s} {notas[i]}")

A Ana ficou com **9.5** (que é da Carla). A Carla ficou com **4.0** (que é do Bruno).

**Nenhum erro vermelho.** O programa rodou feliz e mentiu — o mesmo tipo de bug da ordem dos
`elif` na Aula 3 e do acumulador zerado na Aula 4.

O problema de raiz: a lista guarda **ordem**. Ela não guarda **nome**.

---
# 2. Dicionário: cada valor tem um nome

Colchete vira chave `{}`, e cada item é um par `nome: valor`.

In [ ]:
aluno = {"nome": "Ana", "nota": 8.5, "faltas": 2}

print(aluno)
print(aluno["nome"])
print(aluno["nota"])

Compare com o que vocês vinham fazendo:

| | Como você pede | O que precisa saber |
|---|---|---|
| lista | `alunos[0]` | **em que posição** está |
| dicionário | `aluno["nome"]` | **o nome** da informação |

`aluno["nota"]` continua funcionando mesmo que a ordem mude, mesmo que apareçam dez campos
novos, mesmo que alguém ordene qualquer coisa. **O valor está amarrado ao nome, não à posição.**

Vocabulário: cada nome é uma **chave** (*key*), cada informação é um **valor** (*value*).

In [ ]:
aluno["nota"] = 9.0                       # muda um valor
aluno["email"] = "ana@exemplo.com"        # cria uma chave nova

print(aluno)
print(len(aluno))
print("nota" in aluno)       # o mesmo `in` da Aula 3

Repare: **atribuir numa chave que não existe cria a chave.** Com lista, `notas[10] = 5` numa
lista de 3 itens dá `IndexError`. Aqui, não — o dicionário simplesmente cresce.

### 💥 Quebrando de propósito nº 1

Peça uma chave que não existe:

In [ ]:
print(aluno["telefone"])

```
KeyError: 'telefone'
```

**`KeyError` é o `IndexError` do dicionário.** Quer dizer: *essa chave não existe aqui*.

E existe um jeito de perguntar sem tomar erro:

In [ ]:
print(aluno.get("telefone"))                      # None, sem quebrar
print(aluno.get("telefone", "não informado"))     # com um padrão

> `.get()` é o que você usa quando **não tem certeza** se a chave está lá. E isso é a regra, não
> a exceção, quando o dado vem de fora — de um arquivo, de uma API, de um formulário.

### 💥 Quebrando de propósito nº 2

In [ ]:
print(aluno[0])

```
KeyError: 0
```

**Dicionário não tem índice.** O `[0]` não é "o primeiro item" — é uma tentativa de buscar a
chave `0`, que não existe.

Quando você vê `[ ]` em Python, olhe o que está **dentro**:
- número → é posição, e a coisa é uma lista
- texto → é nome, e a coisa é um dicionário

---
# 3. Percorrer um dicionário

In [ ]:
for chave in aluno:
    print(chave, "->", aluno[chave])

O `for` sozinho passa pelas **chaves**. Existem três jeitos de pedir explicitamente:

In [ ]:
print(list(aluno.keys()))     # só os nomes
print(list(aluno.values()))   # só os valores

for chave, valor in aluno.items():    # os dois de uma vez
    print(f"{chave:8s} = {valor}")

`.items()` é o que você mais vai usar. Ele entrega **duas** variáveis por volta — repare que o
`for` agora tem dois nomes antes do `in`.

---
# 4. Lista de dicionários — a estrutura do mundo real

Esta combinação é, sem exagero, o formato de **quase todo dado** que você vai encontrar:
uma lista, e cada item é um dicionário.

In [ ]:
turma = [
    {"nome": "Ana",   "nota": 8.5, "faltas": 2},
    {"nome": "Bruno", "nota": 4.0, "faltas": 7},
    {"nome": "Carla", "nota": 9.5, "faltas": 0},
]

for a in turma:
    print(f"{a['nome']:6s} {a['nota']:4.1f}  {a['faltas']} faltas")

Repare na aspa: fora é `f"..."`, dentro é `a['nome']` com aspa simples. Se usar a mesma dos dois
lados, o Python se perde.

E agora as funções da Aula 6 voltam, trabalhando em cima disso:

In [ ]:
def media_da_turma(turma):
    soma = 0
    for a in turma:
        soma += a["nota"]
    return soma / len(turma)


def aprovados(turma, corte=7):
    lista = []
    for a in turma:
        if a["nota"] >= corte:
            lista.append(a["nome"])
    return lista


print(f"média: {media_da_turma(turma):.2f}")
print("aprovados:", aprovados(turma))

### E agora o teste que importa

Embaralhe a turma inteira e veja se alguém troca de nota:

In [ ]:
import random

random.seed(3)
random.shuffle(turma)

for a in turma:
    print(f"{a['nome']:6s} {a['nota']}")

A ordem mudou. **As notas foram junto com os nomes.**

Era exatamente isso que estava quebrado no começo da aula.

### 🔧 Exercício

Com a `turma` acima:

1. escreva `buscar(turma, nome)` que devolve o dicionário do aluno — ou `None` se não achar
2. escreva `mais_faltoso(turma)` que devolve o **nome** de quem tem mais faltas
3. acrescente um aluno novo na turma com `.append()`

In [ ]:
# sua vez

---
# 5. JSON: o dicionário virando texto

Um dicionário só existe **dentro** do Python, na memória. Para mandar pela internet ou salvar
num arquivo, ele precisa virar **texto**.

O formato universal para isso chama-se **JSON**. E ele é, na prática, um dicionário escrito.

In [ ]:
import json

aluno = {"nome": "Ana", "nota": 8.5, "aprovado": True, "faltas": 2}

texto = json.dumps(aluno)
print(texto)
print(type(texto))

```
{"nome": "Ana", "nota": 8.5, "aprovado": true, "faltas": 2}
<class 'str'>
```

Parece igual, mas **é uma string** — dá para mandar por rede, salvar em arquivo, colar no
WhatsApp.

Repare no detalhe: o `True` do Python virou `true`, minúsculo. JSON tem as regras dele.

E o caminho de volta:

In [ ]:
de_volta = json.loads(texto)

print(de_volta["nome"])
print(type(de_volta))

> **`dumps` escreve, `loads` lê.** O `s` no fim é de *string*.
>
> `dict → json.dumps → texto → json.loads → dict`

Para ler com olho humano, use `indent`:

In [ ]:
print(json.dumps(aluno, indent=2, ensure_ascii=False))

O `ensure_ascii=False` mantém os acentos legíveis. Sem ele, `ção` vira `\u00e7\u00e3o`.

### 💥 Quebrando de propósito nº 3

In [ ]:
json.loads("{'nome': 'Ana'}")

```
JSONDecodeError: Expecting property name enclosed in double quotes
```

**JSON exige aspas duplas.** Aspa simples funciona em Python, não funciona em JSON.

Esse erro vai acontecer com vocês, e quase sempre a causa é a mesma: alguém colou um dicionário
**Python** onde o programa esperava um **JSON**. São parecidos de propósito, e diferentes o
suficiente para dar dor de cabeça.

As três diferenças que pegam:

In [ ]:
print(json.dumps({"ok": True, "erro": None}))     # Python -> JSON
print(json.loads('{"ok": true, "erro": null}'))   # JSON -> Python

| Python | JSON |
|---|---|
| `True` / `False` | `true` / `false` |
| `None` | `null` |
| aspas simples ou duplas | **só duplas** |

---
# 💊 Pílula de IA — toda API de IA fala JSON

## Etapa 1 — a resposta de um modelo é isto

Quando um programa pede algo a um modelo de IA, o que volta pela internet é **texto no formato
JSON**. Esta é a cara dele:

In [ ]:
resposta_texto = '''
{
  "id": "chatcmpl-9xY2",
  "object": "chat.completion",
  "model": "um-modelo-qualquer",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "Uma lista guarda ordem; um dicionario guarda nome."
      },
      "finish_reason": "stop"
    }
  ],
  "usage": {
    "prompt_tokens": 24,
    "completion_tokens": 11,
    "total_tokens": 35
  }
}
'''

print(type(resposta_texto))

Neste momento isso é **texto**. Uma string gigante e inútil.

## Etapa 2 — uma linha, e vira dicionário

In [ ]:
r = json.loads(resposta_texto)

print(type(r))
print(list(r.keys()))

## Etapa 3 — agora navegue até o texto

Olhe com calma. É lista dentro de dicionário dentro de dicionário — e você aprendeu **as duas
coisas** hoje e sábado passado.

In [ ]:
print(type(r["choices"]))                       # uma LISTA
print(len(r["choices"]))                        # com 1 item
print(list(r["choices"][0].keys()))             # o item é um DICIONÁRIO
print(list(r["choices"][0]["message"].keys()))  # que tem outro dentro

In [ ]:
print(r["choices"][0]["message"]["content"])

```
Uma lista guarda ordem; um dicionario guarda nome.
```

**Aquilo ali é a resposta da IA.** Aquela linha comprida que você vê em todo tutorial de
inteligência artificial:

```python
resposta["choices"][0]["message"]["content"]
```

Ela não tem mistério nenhum. Leia da esquerda para a direita:

- `["choices"]` → a chave *choices*, que é uma **lista**
- `[0]` → o **primeiro** item dela (o modelo pode devolver várias respostas)
- `["message"]` → a chave *message*, um dicionário
- `["content"]` → e finalmente o texto

## Etapa 4 — e o `usage` é a Aula 5 cobrando a conta

In [ ]:
print(r["usage"])
print()
print("tokens que EU mandei :", r["usage"]["prompt_tokens"])
print("tokens que ELE gerou :", r["usage"]["completion_tokens"])
print("total                :", r["usage"]["total_tokens"])

In [ ]:
preco_por_milhao = 3.00      # exemplo — os preços mudam o tempo todo

custo = r["usage"]["total_tokens"] / 1_000_000 * preco_por_milhao
print(f"esta chamada: US$ {custo:.6f}")
print(f"mil chamadas: US$ {custo * 1000:.2f}")

São os **tokens da Aula 5**, os mesmos que fizeram `gato` virar `['g', 'ato']`. É por isso que
português custa mais caro: mais tokens na mesma frase.

## Etapa 5 — e o que você MANDA também é JSON

Não é só a resposta. O pedido também:

In [ ]:
pedido = {
    "model": "um-modelo-qualquer",
    "messages": [
        {"role": "system", "content": "Voce e um professor de programacao."},
        {"role": "user", "content": "O que e um dicionario em Python?"}
    ],
    "temperature": 0.7
}

print(json.dumps(pedido, indent=2, ensure_ascii=False))

Aquele `role` / `content` que aparece em **todo** tutorial de IA é isto aqui: uma **lista de
dicionários**, exatamente a estrutura da turma lá em cima.

- `"role": "system"` → a instrução de fundo
- `"role": "user"` → o que a pessoa perguntou
- `"role": "assistant"` → o que o modelo respondeu antes

E é assim que o modelo "lembra" da conversa: **ele não lembra.** Você reenvia a lista inteira
toda vez. Por isso conversa longa custa mais caro — a lista cresce, e com ela os
`prompt_tokens`.

---

### O que fica da pílula de hoje

1. **API de IA é JSON entrando e JSON saindo.** Você já sabe ler e escrever os dois.
2. **Aquela linha assustadora é só chave, índice, chave, chave.** Sem mistério.
3. **O modelo não tem memória.** O histórico é uma lista de dicionários que *você* reenvia.

Falta uma peça só para vocês chamarem uma API de verdade: como mandar isso pela internet.
Isso é a **Aula 12**.

---
# 🏠 Tarefa da semana

### 1. Agenda de contatos
Uma lista de dicionários com `nome`, `telefone` e `cidade`. Escreva `buscar(agenda, nome)` e
`de_cidade(agenda, cidade)`. Use `.get()` onde o campo pode faltar.

### 2. Refatorar o boletim (de novo)
Pegue o boletim da Aula 5 e troque as duas listas paralelas por **uma lista de dicionários**.
Depois embaralhe e confirme que nada se perde.

### 3. Ler a resposta da IA
Cole o `resposta_texto` da pílula num programa seu e imprima, em três linhas separadas:
o texto da resposta, o modelo usado, e o total de tokens.

### 4. Desafio
Escreva `contar_palavras(frase)` que devolve um **dicionário** com cada palavra e quantas vezes
ela aparece.

*(Dica: `contagem[p] = contagem.get(p, 0) + 1` — é o acumulador da Aula 4 com `.get()`.)*

Isso tem nome, e é o coração do detector de spam da Aula 3.

---

### 🤔 Para pensar até sábado

Feche o Colab agora e abra de novo.

Cadê a sua agenda de contatos? Cadê a turma? Cadê tudo que você digitou?

**Sumiu.** Tudo que a gente fez em sete aulas viveu só na memória, e memória morre quando o
programa fecha.

Um dicionário lindo que desaparece toda vez não serve para nada real. Como é que o dado
**sobrevive**?

*(A resposta é a Aula 8: arquivos. E é a aula em que o Python sai do navegador e entra na
máquina de vocês.)*